In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
df = pd.read_csv('/kaggle/input/datasets/organizations/mlg-ulb/creditcardfraud/creditcard.csv')

#Read the CSV file into a dataframe

df.head()

#display the first 5 rows

In [ ]:
print("Dataset Dimensions (Rows, Columns):")
print(df.shape)

#check dimensions(rows and columns)

print("\nTotal Missing Values:")
print(df.isnull().sum().sum())

#check missing values in the dataset

print("\nClass Count (0 = Normal, 1 = Fraud):")
print(df['Class'].value_counts())

#check normal vs fraud transactions

print("\nClass Breakdown (%):")
print(df['Class'].value_counts(normalize=True) * 100)

#check percentage split

In [ ]:
df.groupby('Class')['Amount'].describe()

#compare mean, median, max for normal vs fraud

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14,5))

#side by side plot for comparison for distribution amount of normal vs fraud transactions

sns.histplot(df[df['Class'] == 0]['Amount'], ax=ax1, bins=50, color='green', kde=True)
ax1.set_title('Normal Transaction (Amount)')
ax1.set_yscale('log')

#plot normal transaction
#using log scale to better see the distribution

sns.histplot(df[df['Class'] == 1]['Amount'], ax=ax2, bins=50, color='red', kde=True)
ax2.set_title('Fraudulent Transaction (Amount)')
ax2.set_yscale('log')

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

df['Hour'] = (df['Time'] / 3600) % 24

#convert seconds to hours of the day (0 to 23)

plt.figure(figsize=(12,5))

#side by side plot for hourly transactions

sns.kdeplot(df[df['Class'] == 0]['Hour'], label='Normal', color='green', fill=True)
sns.kdeplot(df[df['Class'] == 1]['Hour'], label='Fraud', color='red', fill=True)

#plot normal vs fraud transaction hourly

plt.title('Transaction Density by Hour of the Day (0 = Midnight')
plt.xlabel('Hour of the Day')
plt.ylabel('Density')
plt.xticks(range(0,25,2))
plt.legend()
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

#separate features (X) and target (y)

X = df.drop(columns=['Class', 'Time'])  #dropping original raw time column
y = df['Class']

#scale amount and hour so all in the same scale

scaler = StandardScaler()
X[['Amount', 'Hour']] = scaler.fit_transform(X[['Amount', 'Hour']])

#perform train test split (80% train, 20% test)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 42, stratify = y
)

print(f"Training set : {X_train.shape[0]} rows")
print(f"Testing set : {X_test.shape[0]} rows")

# 4. Train a Baseline Logistic Regression Model with class_weight='balanced'
# 'class_weight=balanced' automatically adjusts weights inversely proportional to class frequencies

model = LogisticRegression(class_weight = 'balanced', random_state = 42, max_iter = 1000)
model.fit(X_train, y_train)

y_pred = model.predict (X_test)

print("\n--- Model Classification Report ---")
print(classification_report(y_test, y_pred, target_names=['Normal (0)', 'Fraud (1)']))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

#generate confusion matrix
cm = confusion_matrix(y_test, y_pred)

#plot heatmap to see all the TP, TN, FN, FP
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt= 'd', cmap='Blues',
           xticklabels=['Normal = 0', 'Fraud = 1'],
           yticklabels=['Normal = 0', 'Fraud = 1'])

plt.title('Confusion Matrix - Baseline Logistic Regression')
plt.xlabel('Predicter Label')
plt.ylabel('True Label')
plt.show()

In [ ]:
from sklearn.ensemble import RandomForestClassifier

#initialize Random Forest (n_jobs=-1 uses all CPU cores to speed it up)
rf_model = RandomForestClassifier(n_estimators = 100, random_state = 42, n_jobs = -1)

#fit the model on training data
rf_model.fit(X_train, y_train)

#predict on test data
y_pred_rf = rf_model.predict(X_test)

#classification report
print("\n--- Random Forest Classification Report ---")
print(classification_report(y_test, y_pred_rf, target_names = ['Normal = 0', 'Fraud = 1']))

#confusion matrix
cm_rf = confusion_matrix (y_test, y_pred_rf)
plt.figure(figsize=(6,5))

sns.heatmap(cm_rf, annot=True, fmt='d', cmap = 'Greens',
            xticklabels=['Normal = 0', 'Fraud = 1'],
            yticklabels=['Normal = 0', 'Fraud = 1'])

plt.title("Confusion Matrix - Random Forest")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

# get predicted probabilities for Fraud (Class 1) instead of hard 0 or 1 predictions
y_probs = rf_model.predict_proba(X_test)[:,1]

#testing different decision threshold
thresholds = [0.50, 0.30, 0.20, 0.10]
results = []

for t in thresholds:
    #flag as fraud is probability is above threshold 't'
    y_pred_custom = (y_probs >= t).astype (int)

    cm_custom = confusion_matrix (y_test, y_pred_custom)
    tn, fp, fn, tp = cm_custom.ravel()

    results.append({
        'Threshold' : t,
        'Caught Fraud (TP)' : tp,
        'Missed Fraud (FN)' : fn,
        'False Alarm (FP)' : fp,
        'Precision' : round(precision_score(y_test, y_pred_custom), 2),
        'Recall' : round(recall_score(y_test, y_pred_custom), 2),
        'F1 Score' : round(f1_score(y_test, y_pred_custom), 2)
})

#convert results into clean table
threshold_df = pd.DataFrame(results)
print(threshold_df.to_string (index=False))